# Reproducing Figure xx: GRABACh3.0 Acetylcholine Biosensor Analysis

This notebook reproduces Figure x from **Belal et al. 2025** TITLE TBC.

**Dataset**: DANDI:xxxxx - Title TBC

**Analysis approach**:
- **Biosensor**: GRABACh3.0 genetically encoded acetylcholine indicator
- **Stimulation**: Single-pulse electrical stimulation to evoke acetylcholine release


In [1]:
import json, os, re, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pynwb import NWBHDF5IO

sys.path.append(str(Path.cwd() / "Python functions"))
from master_functions import *


def parse_filename_meta(nwb_path):
    stem = Path(nwb_path).stem
    m = re.match(
        r"sub-(\w+)_animal(\d+)_slice(\d+)_roi(\d+)_ses-(\d{8})T(\d{6})_run-(\d+)_ophys",
        stem,
    )
    if not m:
        return {}
    group, animal, sl, roi, date, time_, run = m.groups()
    return {
        "date":        date,
        "animal_id":   int(animal),
        "group":       group,
        "slice_id":    int(sl),
        "roi_id":      int(roi),
        "slice_label": f"slice{int(sl)}ROI{int(roi)}",
        "run_id":      int(run),
    }


def load_nwb_fluorescence(nwb_path, json_meta=None):
    nwb_path = Path(nwb_path)
    records  = []
    with NWBHDF5IO(str(nwb_path), mode="r", load_namespaces=True) as io:
        nwbfile = io.read()
        meta = {}
        json_meta = json_meta or {}
        meta["nwb_file"]   = nwb_path.name
        meta["key"]        = json_meta.get("dandi_path", nwbfile.identifier)
        meta["nwb_identifier"] = nwbfile.identifier
        meta.update(json_meta)
        meta["session_id"] = nwbfile.session_id
        meta["project"]    = nwbfile.session_id.split("/")[0] if nwbfile.session_id else None
        if nwbfile.subject is not None:
            meta["subject_id"] = nwbfile.subject.subject_id
        meta.update({k: v for k, v in parse_filename_meta(nwb_path).items()
                     if k not in meta})

        trials_df = nwbfile.trials.to_dataframe() if nwbfile.trials is not None else None

        if "ophys" not in nwbfile.processing:
            print(f"[WARN] no ophys in {nwb_path.name}")
            return records

        fluor = nwbfile.processing["ophys"]["Fluorescence"]
        for series_name, series in fluor.roi_response_series.items():
            fluorescence = series.data[:]
            timestamps   = series.get_timestamps()

            stim_time = None
            if trials_df is not None and "roi_series_name" in trials_df.columns:
                match = trials_df[trials_df["roi_series_name"] == series_name]
                if not match.empty:
                    stim_time = float(match.iloc[0]["stimulus_start_time"])
                    for col in ("treatment", "stimulation", "group"):
                        if col in match.columns:
                            meta[col] = match.iloc[0][col]

            records.append({
                **meta,
                "series_name":  series_name,
                "fluorescence": fluorescence,
                "timestamps":   timestamps,
                "stim_time":    stim_time,
                "rate_hz":      getattr(series, "rate", None),
                "n_frames":     len(fluorescence),
            })
    return records


def load_dataset_description_index(nwb_dir):
    desc_path = Path(nwb_dir) / "dataset_description" / "dataset_description.json"
    with open(desc_path, "r") as f:
        desc = json.load(f)

    index = {}
    for figure, entries in desc.get("FigureMappings", {}).items():
        for entry in entries:
            dandi_path = entry["dandi_path"]
            index[Path(nwb_dir) / dandi_path] = {
                "key": dandi_path,
                "figure": figure,
                "original_path": entry.get("original_path"),
                "dandi_path": dandi_path,
            }
    return index


def load_all_nwb(nwb_dir):
    nwb_dir   = Path(nwb_dir)
    nwb_index = load_dataset_description_index(nwb_dir)
    nwb_files = sorted(path for path in nwb_index if path.exists() and path.name.endswith("_ophys.nwb"))
    if not nwb_files:
        print(f"No .nwb files found under {nwb_dir}/**/sub-*/")
        return pd.DataFrame()
    all_records = []
    for f in nwb_files:
        print(f"  loading {f.name} …", end=" ")
        recs = load_nwb_fluorescence(f, nwb_index[f])
        print(f"{len(recs)} series")
        all_records.extend(recs)
    df = pd.DataFrame(all_records)
    print(f"\n→ {len(df)} total records from {len(nwb_files)} files")
    return df


NWB_DIR = Path.cwd() / "NWBdata"

df = load_all_nwb(NWB_DIR)

# preview metadata columns (no arrays)
print(df[["nwb_file", "group", "animal_id", "slice_id", "roi_id",
          "run_id", "series_name", "n_frames", "stim_time"]])

# access one trace
row = df.iloc[0]
print("timestamps (first 5):", row["timestamps"][:5])
print("fluorescence (first 5):", row["fluorescence"][:5])




ModuleNotFoundError: No module named 'surmeier_lab_to_nwb'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

df_grab = df[df["series_name"].str.contains("GRABCh")].reset_index(drop=True)
ii = 160
trial = df_grab.iloc[ii]
t = np.asarray(trial["timestamps"], dtype=float)
f_raw = np.asarray(trial["fluorescence"], dtype=float)
stim_time = float(trial["stim_time"]) - t[0]

print(trial[["nwb_file", "group", "animal_id", "slice_id", "roi_id", "run_id"]])
print(f"stim_time={stim_time:.2f}s  n_frames={len(t)}")

pmt_background  = 144
baseline_window = 1
xlim = [4.5, 7]

dff = compute_dff(f_raw=f_raw, time=t, stim_time=stim_time, pmt_background=pmt_background, baseline_window=baseline_window)
plot_dff(time=t, dff=dff, stim_time=stim_time, xlim=xlim, lwd=1.0, tick_len=4, axis_color="black")


In [ ]:

rows = []
for _, trial in df_grab.iterrows():
    t         = np.asarray(trial["timestamps"], dtype=float)
    f_raw     = np.asarray(trial["fluorescence"], dtype=float)
    stim_time = float(trial["stim_time"]) - t[0]
    dff       = compute_dff(f_raw=f_raw, time=t, stim_time=stim_time,
                            pmt_background=pmt_background, baseline_window=baseline_window)
    row       = trial.to_dict()
    row["fluorescence"] = dff
    rows.append(row)

df_fl = pd.DataFrame(rows).reset_index(drop=True)

In [ ]:
ii=160
trial = df_fl.iloc[ii]

print(trial[["nwb_file","group","animal_id","slice_id","roi_id","run_id"]])
print(f"stim_time={stim_time:.2f}s  n_frames={len(t)}")

t = np.asarray(trial["timestamps"], dtype=float)           
dff = np.asarray(trial["fluorescence"], dtype=float)
       
plot_dff(time=t, dff=dff, stim_time=stim_time, xlim=xlim, lwd=1.0, tick_len=4, axis_color="black")


In [ ]:
group_cols = ["group", "animal_id", "slice_id", "roi_id"]
rows = []
for keys, grp in df_fl.groupby(group_cols, sort=False):
    traces = [np.asarray(f, dtype=float) for f in grp["fluorescence"].values]
    min_len = min(len(t) for t in traces)
    traces_clipped = np.vstack([t[:min_len] for t in traces])
    dff_avg = traces_clipped.mean(axis=0)
    ref = grp.sort_values("run_id").iloc[0].to_dict()
    ref["fluorescence"] = dff_avg
    ref["timestamps"]   = np.asarray(ref["timestamps"], dtype=float)[:min_len]
    ref["n_reps"]       = len(grp)
    rows.append(ref)
df_fl_average = pd.DataFrame(rows).reset_index(drop=True)




In [ ]:
df_fl_average

In [ ]:
# find peak ΔF/F for each averaged ROI
df_fl_average["fmax"] = df_fl_average["fluorescence"].apply(
    lambda f: np.asarray(f, dtype=float).max()
)

# split by condition
ctrl = df_fl_average[df_fl_average["group"] == "ctrl"]
test = df_fl_average[df_fl_average["group"] == "test"]

print("ctrl fmax:", ctrl["fmax"].describe())
print("test fmax:", test["fmax"].describe())


In [ ]:
fig, ax = plt.subplots()

box_plot = boxplot_rtype(
    ax,
    [ctrl["fmax"].values, test["fmax"].values],
    rtype=7,
    whis="minmax",
    positions=[1, 2],
    patch_artist=True,
    widths=0.25,
    boxprops=dict(linewidth=1.5),
    whiskerprops=dict(linewidth=1.5),
    capprops=dict(linewidth=1.5),
    median_overhang = 0.02,
    medianprops=dict(color="black", linewidth=4),
    showpoints=True,
    jitter_frac=1.2,
    point_diameter=4,
    point_alpha=1,
    point_color="gray",
    point_edgecolor="black",
    paired=False,
)

box_plot["boxes"][0].set_facecolor("white")
box_plot["boxes"][1].set_facecolor("white")

ax.set_xticks([1, 2])
ax.set_xticklabels(["ctrl", "MCI-Park"])
ax.set_ylabel("peak ΔF/F (fmax)")
ax.set_title("Peak fluorescence per ROI")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
ctrl_ids = sorted(df_fl_average[df_fl_average['group'] == 'ctrl']['animal_id'].unique())
test_ids = sorted(df_fl_average[df_fl_average['group'] != 'ctrl']['animal_id'].unique())

n_ctrl = len(ctrl_ids)

# Relabel test animals after ctrl animals
test_relabel = {orig: n_ctrl + i + 1 for i, orig in enumerate(test_ids)}

df_export = df_fl_average.copy()

df_export['Animal'] = df_export.apply(
    lambda r: r['animal_id'] if r['group'] == 'ctrl'
    else test_relabel[r['animal_id']],
    axis=1
)

# Rename condition values
df_export['group'] = df_export['group'].replace({
    'ctrl': 'Control',
    'test': 'MCI-Park'
})

out = (
    df_export.rename(columns={
        'slice_id': 'Slice',
        'roi_id': 'ROI',
        'group': 'Condition',
        'fmax': '(F1-F0)/F0'
    })
    [['Animal', 'Slice', 'ROI', 'Condition', '(F1-F0)/F0']]
    .sort_values(['Animal', 'Slice', 'ROI'])
    .reset_index(drop=True)
)

# out.to_excel(f'{NWB_DIR}/results.xlsx', index=False)

In [ ]:
out

In [ ]:
# single examples
%matplotlib widget

ii=8
trial = df_fl_average.iloc[ii]
xlim = [4.5, 7]
ylim = [-0.05, 0.8]

vmin = 0
vmax = 0.8
print(trial[["nwb_file","group","animal_id","slice_id","roi_id","run_id"]])
print(f"stim_time={stim_time:.2f}s  n_frames={len(t)}")

t = np.asarray(trial["timestamps"], dtype=float)           
dff = np.asarray(trial["fluorescence"], dtype=float)
       
plot_dff(time=t, dff=dff, stim_time=stim_time, xlim=xlim, ylim=ylim, lwd=1.0, tick_len=4, vmin=vmin, vmax=vmax, axis_color="black")

# export single eg
cond = {"ctrl": "Control", "test": "MCI-Park"}.get(trial["group"], trial["group"])

meta_out = pd.DataFrame([{
    "Animal": trial["animal_id"],
    "Slice": trial["slice_id"],
    "ROI": trial["roi_id"],
    "Condition": cond,
    "run_id": trial["run_id"],
    "nwb_file": trial["nwb_file"],
    "stim_time_s": stim_time,
    "(F1-F0)/F0_peak": float(np.nanmax(dff)),
}])

trace_out = pd.DataFrame({
    "time_s": t,
    "time_from_start_s": t - t[0],
    "dff": dff,
})

export_path = Path(NWB_DIR) / f"single_example_animal{trial['animal_id']}_slice{trial['slice_id']}_roi{trial['roi_id']}.xlsx"
with pd.ExcelWriter(export_path, engine="openpyxl") as writer:
    meta_out.to_excel(writer, sheet_name="summary", index=False)
    trace_out.to_excel(writer, sheet_name="trace", index=False)

print(f"Exported: {export_path}")


In [ ]:
ii=30
trial = df_fl_average.iloc[ii]

print(trial[["nwb_file","group","animal_id","slice_id","roi_id", "run_id"]])
print(f"stim_time={stim_time:.2f}s  n_frames={len(t)}")

t = np.asarray(trial["timestamps"], dtype=float)           
dff = np.asarray(trial["fluorescence"], dtype=float)
       
plot_dff(time=t, dff=dff, stim_time=stim_time, xlim=xlim, ylim=ylim, lwd=1.0, tick_len=4, vmin=vmin, vmax=vmax, axis_color="black")

# export single eg
cond = {"ctrl": "Control", "test": "MCI-Park"}.get(trial["group"], trial["group"])

meta_out = pd.DataFrame([{
    "Animal": trial["animal_id"],
    "Slice": trial["slice_id"],
    "ROI": trial["roi_id"],
    "Condition": cond,
    "run_id": trial["run_id"],
    "nwb_file": trial["nwb_file"],
    "stim_time_s": stim_time,
    "(F1-F0)/F0_peak": float(np.nanmax(dff)),
}])

trace_out = pd.DataFrame({
    "time_s": t,
    "time_from_start_s": t - t[0],
    "dff": dff,
})

export_path = Path(NWB_DIR) / f"single_example_animal{trial['animal_id']}_slice{trial['slice_id']}_roi{trial['roi_id']}.xlsx"
with pd.ExcelWriter(export_path, engine="openpyxl") as writer:
    meta_out.to_excel(writer, sheet_name="summary", index=False)
    trace_out.to_excel(writer, sheet_name="trace", index=False)

print(f"Exported: {export_path}")